In [ ]:
# Run this cell to import pyspark and to define start_spark() and stop_spark()

import findspark

findspark.init()

import getpass
import pandas
import pyspark
import random
import re

from IPython.display import display, HTML
from pyspark import SparkContext
from pyspark.sql import SparkSession


# Constants used to interact with Azure Blob Storage using the hdfs command or Spark

global username

username = re.sub('@.*', '', getpass.getuser())


# Functions used below

def dict_to_html(d):
    """Convert a Python dictionary into a two column table for display.
    """

    html = []

    html.append(f'<table width="100%" style="width:100%; font-family: monospace;">')
    for k, v in d.items():
        html.append(f'<tr><td style="text-align:left;">{k}</td><td>{v}</td></tr>')
    html.append(f'</table>')

    return ''.join(html)


def show_as_html(df, n=20):
    """Leverage existing pandas jupyter integration to show a spark dataframe as html.
    
    Args:
        n (int): number of rows to show (default: 20)
    """

    display(df.limit(n).toPandas())

    
def display_spark():
    """Display the status of the active Spark session if one is currently running.
    """
    
    if 'spark' in globals() and 'sc' in globals():

        name = sc.getConf().get("spark.app.name")

        html = [
            f'<p><b>Spark</b></p>',
            f'<p>The spark session is <b><span style="color:green">active</span></b>, look for <code>{name}</code> under the running applications section in the Spark UI.</p>',
            f'<ul>',
            f'<li><a href="http://localhost:{sc.uiWebUrl.split(":")[-1]}" target="_blank">Spark Application UI</a></li>',
            f'</ul>',
            f'<p><b>Config</b></p>',
            dict_to_html({k: v for k, v in sc.getConf().getAll() if not re.search(r"(secret|password|token|credential|sas|account\.key)", k, re.I)}),
            f'<p><b>Notes</b></p>',
            f'<ul>',
            f'<li>The spark session <code>spark</code> and spark context <code>sc</code> global variables have been defined by <code>start_spark()</code>.</li>',
            f'<li>Please run <code>stop_spark()</code> before closing the notebook or restarting the kernel or kill <code>{name}</code> by hand using the link in the Spark UI.</li>',
            f'</ul>',
        ]
        display(HTML(''.join(html)))
        
    else:
        
        html = [
            f'<p><b>Spark</b></p>',
            f'<p>The spark session is <b><span style="color:red">stopped</span></b>, confirm that <code>{username} (notebook)</code> is under the completed applications section in the Spark UI.</p>',
            f'<ul>',
            f'<li><a href="http://mathmadslinux2p.canterbury.ac.nz:8080/" target="_blank">Spark UI</a></li>',
            f'</ul>',
        ]
        display(HTML(''.join(html)))


# Functions to start and stop spark

def start_spark(executor_instances=2, executor_cores=1, worker_memory=1, master_memory=1):
    """Start a new Spark session and define globals for SparkSession (spark) and SparkContext (sc).
    
    Args:
        executor_instances (int): number of executors (default: 2)
        executor_cores (int): number of cores per executor (default: 1)
        worker_memory (float): worker memory (default: 1)
        master_memory (float): master memory (default: 1)
    """

    global spark
    global sc

    cores = executor_instances * executor_cores
    partitions = cores * 4
    port = 4000 + random.randint(1, 999)

    spark = (
        SparkSession.builder
        .config("spark.driver.extraJavaOptions", f"-Dderby.system.home=/tmp/{username}/spark/")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.executor.instances", str(executor_instances))
        .config("spark.executor.cores", str(executor_cores))
        .config("spark.cores.max", str(cores))
        .config("spark.driver.memory", f'{master_memory}g')
        .config("spark.executor.memory", f'{worker_memory}g')
        .config("spark.driver.maxResultSize", "0")
        .config("spark.sql.shuffle.partitions", str(partitions))
        .config("spark.kubernetes.container.image", "madsregistry001.azurecr.io/hadoop-spark:v3.3.5-openjdk-8")
        .config("spark.kubernetes.container.image.pullPolicy", "IfNotPresent")
        .config("spark.kubernetes.memoryOverheadFactor", "0.3")
        .config("spark.memory.fraction", "0.1")
        .config("spark.app.name", f"{username} (notebook)")
        .getOrCreate()
    )
    sc = SparkContext.getOrCreate()
    
    display_spark()

    
def stop_spark():
    """Stop the active Spark session and delete globals for SparkSession (spark) and SparkContext (sc).
    """

    global spark
    global sc

    if 'spark' in globals() and 'sc' in globals():

        spark.stop()

        del spark
        del sc

    display_spark()


# Make css changes to improve spark output readability

html = [
    '<style>',
    'pre { white-space: pre !important; }',
    'table.dataframe td { white-space: nowrap !important; }',
    'table.dataframe thead th:first-child, table.dataframe tbody th { display: none; }',
    '</style>',
]
display(HTML(''.join(html)))

In [ ]:
# Run this cell to start a spark session in this notebook

start_spark(executor_instances=4, executor_cores=4, worker_memory=4, master_memory=1)

In [ ]:
# Spark imports

from pyspark.sql import Row, DataFrame, Window, functions as F
from pyspark.sql.types import *

In [ ]:
!hdfs dfs -ls wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/

In [ ]:
!hdfs dfs -ls -R wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/ > "./data_structure.txt"

In [ ]:
with open('./data_structure.txt') as f:
    lines = f.readlines()

paths = []
for line in lines:
    path = line.strip().split()[-1]
    if 'msd/' in path:
        rel = path.split('msd/', 1)[1].lstrip('/')
        if not ((rel.endswith('.csv.gz') or rel.endswith('.tsv.gz')) and not rel.startswith('main/')):
            paths.append(rel)

tree = {}
for path in paths:
    parts = path.split('/')
    node = tree
    for part in parts:
        node = node.setdefault(part, {})

def print_tree(tree, prefix=''):
    items = list(tree.items())
    for i, (name, sub) in enumerate(items):
        branch = '└── ' if i == len(items) - 1 else '├── '
        print(prefix + branch + name)
        next_prefix = prefix + ('    ' if i == len(items) - 1 else '│   ')
        print_tree(sub, next_prefix)

print('msd/')
print_tree(tree)


In [ ]:
!hdfs dfs -du -h wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/main/

In [ ]:
!hdfs dfs -du -h wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/genre/

In [ ]:
!hdfs dfs -du -h wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/

In [ ]:
!hdfs dfs -du -h wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/tasteprofile/

In [ ]:
!hdfs dfs -du -h wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/tasteprofile/mismatches

In [ ]:
!hdfs dfs -du -h wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes

In [ ]:
!hdfs dfs -du -h wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features

In [ ]:
def count_data_types_from_schema(df):
    """
    Count the number of columns for each data type based on the schema of the DataFrame.
    This function does not require loading any actual data into memory.
    Only the schema metadata is accessed.
    """
    # Get schema fields
    schema_fields = df.schema.fields

    # Dictionary to hold data type counts
    data_type_counts = {}

    # Iterate through schema fields
    for field in schema_fields:
        data_type = field.dataType.typeName()
        if data_type in data_type_counts:
            data_type_counts[data_type] += 1
        else:
            data_type_counts[data_type] = 1

    # Print results
    for data_type, count in data_type_counts.items():
        print(f"{data_type}: {count} columns")


#### Main

In [ ]:
metadata_sample = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/main/metadata.csv.gz",
    header=True,
    inferSchema=True
).limit(100)

# display schema and first few rows
metadata_sample.printSchema()
show_as_html(metadata_sample)

In [ ]:
analysis_sample = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/main/analysis.csv.gz",
    header=True,
    inferSchema=True
).limit(100)

# display schema and first few rows
analysis_sample.printSchema()
show_as_html(analysis_sample)

In [ ]:
# ===== Schema: main/analysis.csv.gz =====
print("\n===== Schema: main/analysis.csv.gz =====")
analysis_df = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/main/analysis.csv.gz",
    header=True,
    inferSchema=True
)
count_data_types_from_schema(analysis_df)
print("Row count:", analysis_df.count())

# ===== Schema: main/metadata.csv.gz =====
print("\n===== Schema: main/metadata.csv.gz =====")
metadata_df = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/main/metadata.csv.gz",
    header=True,
    inferSchema=True
)
count_data_types_from_schema(metadata_df)
print("Row count:", metadata_df.count())

In [ ]:
# Count distinct songs (song_id)
unique_songs = metadata_df.select("song_id").distinct().count()

print(f"Number of Unique songs: {unique_songs}")

#### Genre

In [ ]:
# Load a TSV file
genres_sample = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/genre/msd-MAGD-genreAssignment.tsv",
    header=False,
    inferSchema=True,
    sep="\t"
).limit(100)

genres_sample.printSchema()
show_as_html(genres_sample)

In [ ]:
# ===== Schema: genre/msd-MASD-styleAssignment.tsv =====
print("\n===== Schema: genre/msd-MASD-styleAssignment.tsv =====")
masd_df = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/genre/msd-MASD-styleAssignment.tsv",
    header=False,
    inferSchema=True,
    sep="\t"
)
count_data_types_from_schema(masd_df)
print("Row count:", masd_df.count())

# ===== Schema: genre/msd-topMAGD-genreAssignment.tsv =====
print("\n===== Schema: genre/msd-topMAGD-genreAssignment.tsv =====")
topmagd_df = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/genre/msd-topMAGD-genreAssignment.tsv",
    header=False,
    inferSchema=True,
    sep="\t"
)
count_data_types_from_schema(topmagd_df)
print("Row count:", topmagd_df.count())

# ===== Schema: genre/msd-MAGD-genreAssignment.tsv =====
print("\n===== Schema: genre/msd-MAGD-genreAssignment.tsv =====")
magd_df = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/genre/msd-MAGD-genreAssignment.tsv",
    header=False,
    inferSchema=True,
    sep="\t"
)
count_data_types_from_schema(magd_df)
print("Row count:", magd_df.count())

#### Attribute

In [ ]:
# Load an uncompressed attribute CSV file
attributes_sample = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-jmir-area-of-moments-all-v1.0.attributes.csv",
    header=False,
    inferSchema=True
).limit(100)

attributes_sample.printSchema()
show_as_html(attributes_sample)

In [ ]:
# ===== Schema: audio/attributes/msd-trh-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-trh-v1.0.attributes.csv =====")
trh_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-trh-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(trh_attr)
print("Row count:", trh_attr.count())

# ===== Schema: audio/attributes/msd-jmir-spectral-derivatives-all-all-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-jmir-spectral-derivatives-all-all-v1.0.attributes.csv =====")
spectral_deriv_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-jmir-spectral-derivatives-all-all-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(spectral_deriv_attr)
print("Row count:", spectral_deriv_attr.count())

# ===== Schema: audio/attributes/msd-jmir-spectral-all-all-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-jmir-spectral-all-all-v1.0.attributes.csv =====")
spectral_all_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-jmir-spectral-all-all-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(spectral_all_attr)
print("Row count:", spectral_all_attr.count())

# ===== Schema: audio/attributes/msd-marsyas-timbral-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-marsyas-timbral-v1.0.attributes.csv =====")
marsyas_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-marsyas-timbral-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(marsyas_attr)
print("Row count:", marsyas_attr.count())

# ===== Schema: audio/attributes/msd-jmir-methods-of-moments-all-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-jmir-methods-of-moments-all-v1.0.attributes.csv =====")
methods_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-jmir-methods-of-moments-all-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(methods_attr)
print("Row count:", methods_attr.count())

# ===== Schema: audio/attributes/msd-ssd-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-ssd-v1.0.attributes.csv =====")
ssd_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-ssd-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(ssd_attr)
print("Row count:", ssd_attr.count())

# ===== Schema: audio/attributes/msd-mvd-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-mvd-v1.0.attributes.csv =====")
mvd_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-mvd-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(mvd_attr)
print("Row count:", mvd_attr.count())

# ===== Schema: audio/attributes/msd-rh-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-rh-v1.0.attributes.csv =====")
rh_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-rh-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(rh_attr)
print("Row count:", rh_attr.count())

# ===== Schema: audio/attributes/msd-jmir-lpc-all-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-jmir-lpc-all-v1.0.attributes.csv =====")
lpc_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-jmir-lpc-all-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(lpc_attr)
print("Row count:", lpc_attr.count())

# ===== Schema: audio/attributes/msd-jmir-area-of-moments-all-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-jmir-area-of-moments-all-v1.0.attributes.csv =====")
area_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-jmir-area-of-moments-all-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(area_attr)
print("Row count:", area_attr.count())

# ===== Schema: audio/attributes/msd-tssd-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-tssd-v1.0.attributes.csv =====")
tssd_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-tssd-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(tssd_attr)
print("Row count:", tssd_attr.count())

# ===== Schema: audio/attributes/msd-rp-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-rp-v1.0.attributes.csv =====")
rp_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-rp-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(rp_attr)
print("Row count:", rp_attr.count())

# ===== Schema: audio/attributes/msd-jmir-mfcc-all-v1.0.attributes.csv =====
print("\n===== Schema: audio/attributes/msd-jmir-mfcc-all-v1.0.attributes.csv =====")
mfcc_attr = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-jmir-mfcc-all-v1.0.attributes.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(mfcc_attr)
print("Row count:", mfcc_attr.count())


#### Feature

In [ ]:
# Load an uncompressed feature CSV file
features_sample = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-jmir-area-of-moments-all-v1.0.csv",
    header=False,
    inferSchema=True
).limit(100)

features_sample.printSchema()
show_as_html(features_sample)

In [ ]:
# ===== Schema: audio/features/msd-jmir-area-of-moments-all-v1.0.csv =====
print("\n===== Schema: audio/features/msd-jmir-area-of-moments-all-v1.0.csv =====")
area_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-jmir-area-of-moments-all-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(area_feat)
print("Row count:", area_feat.count())

# ===== Schema: audio/features/msd-tssd-v1.0.csv =====
print("\n===== Schema: audio/features/msd-tssd-v1.0.csv =====")
tssd_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-tssd-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(tssd_feat)
print("Row count:", tssd_feat.count())

# ===== Schema: audio/features/msd-ssd-v1.0.csv =====
print("\n===== Schema: audio/features/msd-ssd-v1.0.csv =====")
ssd_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-ssd-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(ssd_feat)
print("Row count:", ssd_feat.count())

# ===== Schema: audio/features/msd-marsyas-timbral-v1.0.csv =====
print("\n===== Schema: audio/features/msd-marsyas-timbral-v1.0.csv =====")
marsyas_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-marsyas-timbral-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(marsyas_feat)
print("Row count:", marsyas_feat.count())

# ===== Schema: audio/features/msd-rh-v1.0.csv =====
print("\n===== Schema: audio/features/msd-rh-v1.0.csv =====")
rh_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-rh-v1.0.csv",
    header=False,
    inferSchema=True
)

count_data_types_from_schema(rh_feat)
print("Row count:", rh_feat.count())

# ===== Schema: audio/features/msd-jmir-spectral-derivatives-all-all-v1.0.csv =====
print("\n===== Schema: audio/features/msd-jmir-spectral-derivatives-all-all-v1.0.csv =====")
spectral_deriv_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-jmir-spectral-derivatives-all-all-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(spectral_deriv_feat)
print("Row count:", spectral_deriv_feat.count())

# ===== Schema: audio/features/msd-jmir-spectral-all-all-v1.0.csv =====
print("\n===== Schema: audio/features/msd-jmir-spectral-all-all-v1.0.csv =====")
spectral_all_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-jmir-spectral-all-all-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(spectral_all_feat)
print("Row count:", spectral_all_feat.count())

# ===== Schema: audio/features/msd-trh-v1.0.csv =====
print("\n===== Schema: audio/features/msd-trh-v1.0.csv =====")
trh_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-trh-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(trh_feat)
print("Row count:", trh_feat.count())

# ===== Schema: audio/features/msd-jmir-lpc-all-v1.0.csv =====
print("\n===== Schema: audio/features/msd-jmir-lpc-all-v1.0.csv =====")
lpc_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-jmir-lpc-all-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(lpc_feat)
print("Row count:", lpc_feat.count())

# ===== Schema: audio/features/msd-jmir-methods-of-moments-all-v1.0.csv =====
print("\n===== Schema: audio/features/msd-jmir-methods-of-moments-all-v1.0.csv =====")
methods_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-jmir-methods-of-moments-all-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(methods_feat)
print("Row count:", methods_feat.count())

# ===== Schema: audio/features/msd-jmir-mfcc-all-v1.0.csv =====
print("\n===== Schema: audio/features/msd-jmir-mfcc-all-v1.0.csv =====")
mfcc_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-jmir-mfcc-all-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(mfcc_feat)
print("Row count:", mfcc_feat.count())

# ===== Schema: audio/features/msd-mvd-v1.0.csv =====
print("\n===== Schema: audio/features/msd-mvd-v1.0.csv =====")
mvd_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-mvd-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(mvd_feat)
print("Row count:", mvd_feat.count())

# ===== Schema: audio/features/msd-rp-v1.0.csv =====
print("\n===== Schema: audio/features/msd-rp-v1.0.csv =====")
rp_feat = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/msd-rp-v1.0.csv",
    header=False,
    inferSchema=True
)
count_data_types_from_schema(rp_feat)
print("Row count:", rp_feat.count())


#### tasteprofile

In [ ]:
# Load a TSV file
triplets_sample = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/tasteprofile/triplets.tsv",
    header=False,
    inferSchema=True,
    sep="\t"
).limit(100)

triplets_sample.printSchema()
show_as_html(triplets_sample)

In [ ]:
# ===== Schema: tasteprofile/triplets.tsv =====
print("\n===== Schema: tasteprofile/triplets.tsv =====")
triplets_df = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/tasteprofile/triplets.tsv",
    header=False,
    inferSchema=True,
    sep="\t"
)
count_data_types_from_schema(triplets_df)
print("Row count:", triplets_df.count())

In [ ]:
# ===== Schema: tasteprofile/mismatches/sid_mismatches.txt =====
print("\n===== Schema: tasteprofile/mismatches/sid_mismatches.txt =====")
sid_mismatches_df = spark.read.text(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/tasteprofile/mismatches/sid_mismatches.txt",
)
sid_mismatches_df.printSchema()
count_data_types_from_schema(sid_mismatches_df)
print("Row count:", sid_mismatches_df.count())

# ===== Schema: tasteprofile/mismatches/sid_matches_manually_accepted.txt =====
print("\n===== Schema: tasteprofile/mismatches/sid_matches_manually_accepted.txt =====")
sid_matches_df = spark.read.text(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/tasteprofile/mismatches/sid_matches_manually_accepted.txt",
)
sid_matches_df.printSchema()
count_data_types_from_schema(sid_matches_df)
print("Row count:", sid_matches_df.count())


## Q2

#### a)

In [ ]:
# Load an uncompressed attribute CSV file
attributes_sample = spark.read.csv(
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/msd-rp-v1.0.attributes.csv",
    header=False,
    inferSchema=True
).limit(100)

attributes_sample.printSchema()
show_as_html(attributes_sample)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, FloatType

# Mapping from attribute type string to Spark DataType
type_mapping = {
    "string": StringType(),
    "numeric": DoubleType(),
    "real": DoubleType(),
    "float": DoubleType(),
}

def get_schema_from_attributes(prefix):
    """
    Read the attributes file for `prefix` and build a StructType
    for the matching features file.
    """
    attr_path = f"wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/{prefix}.attributes.csv"
    # Load attribute names and types
    attr_df = spark.read.csv(attr_path, header=False, inferSchema=False)
    attrs = attr_df.collect()

    fields = []
    # Map each attribute row to a StructField
    for row in attrs:
        name = row[0]
        dtype_key = row[1].lower()
        spark_type = type_mapping.get(dtype_key, StringType())
        fields.append(StructField(name, spark_type, True))

    return StructType(fields)

# List all 13 attribute prefixes
attribute_prefixes = [
    "msd-trh-v1.0",
    "msd-jmir-spectral-derivatives-all-all-v1.0",
    "msd-jmir-spectral-all-all-v1.0",
    "msd-marsyas-timbral-v1.0",
    "msd-jmir-methods-of-moments-all-v1.0",
    "msd-ssd-v1.0",
    "msd-mvd-v1.0",
    "msd-rh-v1.0",
    "msd-jmir-lpc-all-v1.0",
    "msd-jmir-area-of-moments-all-v1.0",
    "msd-tssd-v1.0",
    "msd-rp-v1.0",
    "msd-jmir-mfcc-all-v1.0"
]

# Generate and save schema for each attribute file
schema_map = {}
for prefix in attribute_prefixes:
    schema_map[prefix] = get_schema_from_attributes(prefix)

#### b)

In [ ]:
# Choose one feature prefix
prefix = "msd-jmir-mfcc-all-v1.0"

# Retrieve the schema generated in step (a)
schema = schema_map[prefix]

# Path to the feature files (no header row)
feature_path = f"wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/features/{prefix}.csv"

# Load the feature dataset using the predefined schema
jmir_mfcc_df = (
    spark.read
         .csv(feature_path, schema=schema, header=False)
)

# Verify
jmir_mfcc_df.printSchema() 
jmir_mfcc_df.show(5)         


#### c)

In [ ]:
# jmir_mfcc_df.columns

In [ ]:

attributes_sample = (
    spark.read
         .csv(
             "wasbs://campus-data@madsstorage002.blob.core.windows.net/"
             "msd/audio/attributes/msd-marsyas-timbral-v1.0.attributes.csv",
             header=False,
             inferSchema=True
         )
         .limit(100)
)

# 2) Show all rows without truncation
attributes_sample.show(n=100, truncate=False)

# 3) Or show a few rows in vertical format for readability
attributes_sample.show(n=5, truncate=False, vertical=True)


In [ ]:

# Abbreviation mapping for key terms
abbr = {
    "mean":              "mean",
    "standard_deviation": "std",
    "average":            "avg",
    "overall":            "ovr",
    "component":            "comp",
    "spectral_centroid":            "cent",
    "spectral":            "spec",
    "zero_crossings":            "zcr",
    "area_method_of_moments":            "amom",
    "method_of_moments":            "mom",
    "moments":            "mon",
    "root_mean_square":            "rms",
    "spectral_variability":            "variab",
}

def rename(col_name: str, idx: int, prefix_short: str) -> str:
    """
    1. Lowercase & replace non-alphanumerics with underscores.
    2. Substitute full words with abbreviations.
    3. Split into tokens, then:
       - Keep each abbr token wrapped in underscores.
       - Group other tokens: take first letters, merge into one string, wrap in underscores.
    4. Remove duplicate segments, prepend prefix_short, and append an index suffix.
    5. Enforce max length of 30 chars (including suffix).
    """
    # 1. normalize
    s = re.sub(r'[^A-Za-z0-9]+', '_', col_name.lower()).strip('_')
    # 2. replace with abbreviations
    for full, short in abbr.items():
        s = s.replace(full, short)
    tokens = s.split('_')

    parts, i = [], 0
    abbr_vals = set(abbr.values())
    # 3. process tokens
    while i < len(tokens):
        t = tokens[i]
        if t in abbr_vals:
            parts.append(f"_{t}_")
            i += 1
        else:
            # collect a run of non-abbr tokens
            letters = []
            while i < len(tokens) and tokens[i] not in abbr_vals:
                letters.append(tokens[i][0])
                i += 1
            parts.append(f"_{''.join(letters)}_")

    # remove duplicates while preserving order
    seen = set()
    uniq = []
    for seg in parts:
        if seg not in seen:
            uniq.append(seg)
            seen.add(seg)

    core = "".join(uniq)
    suffix = f"_{idx}"
    max_len = 30
    base = (prefix_short + core)[: max_len - len(suffix)]
    return f"{base}{suffix}"

# Process all datasets
dfs = {}
for prefix in attribute_prefixes:
    # derive short prefix
    p_short = "_".join(prefix.split("-")[1:3])
    schema = schema_map[prefix]
    path = (
        "wasbs://campus-data@madsstorage002.blob.core.windows.net"
        f"/msd/audio/features/{prefix}.csv"
    )

    # load without header
    df = spark.read.csv(path, schema=schema, header=False)

    # rename columns with index
    new_names = [
        rename(col, idx, p_short)
        for idx, col in enumerate(df.columns, start=1)
    ]
    df = df.toDF(*new_names)

    # store & inspect
    dfs[prefix] = df
    df.printSchema()
    df.show(3)


In [ ]:
stop_spark()